# DDR diabetic retinopathy: Kaggle notebook (one track per notebook)

Runs Phases 1-5 of the project for **one track**: install, inspect, explore, preprocess + freeze splits, train every strategy, compare on validation.
It deliberately **stops before the test split** (Phase 6): you pick the winner first, write it down, then evaluate once (see the last cell).

**Before you press Run All**
1. Right panel -> Session options: **Internet = On**, **Accelerator = GPU (T4 or P100)**.
2. **Add Input**: for `TRACK = "grading"` add `mariaherrerot/ddrdataset`; for `TRACK = "segmentation"` add `sunfish141/ddr-segmentation`.
3. Edit the settings cell below (at least `TRACK`). Use a *separate copy of this notebook* for the other track.
4. First time? Set `QUICK_TEST = True` to check everything works in minutes (1 epoch per run, results thrown into a scratch folder). Then set it to `False`.
5. For the real run use **Save Version -> Save & Run All (Commit)**: it keeps running in the background and saves `/kaggle/working` as the notebook output.

Resuming after a session limit: add the previous version of this notebook as an input (**Add Input -> Notebook Output**) and Run All again.
Frozen splits, finished runs and the preprocessed cache are restored from it, and finished runs are skipped.

In [ ]:
# ---------------------------- SETTINGS (edit me) ----------------------------
TRACK = "grading"          # "grading" or "segmentation"
QUICK_TEST = True          # True: 1 epoch per run, scratch output folder. Set False for the real run.
BRANCH = "claude/sweet-sagan-8khcuq"   # repo branch to use (change to "main" once merged)
NUM_WORKERS = 2            # data-loader processes; Kaggle has few CPU cores. Does not change results.
PRETRAINED = True          # ImageNet weights (needs Internet = On)
DATA_SRC = None            # None = auto-detect under /kaggle/input; or give the folder path yourself
REPO_URL = "https://github.com/kazimab1/DDR-DR"

In [ ]:
# ---------------------------- SETUP: helper, code, libraries ----------------------------
import glob, os, shutil, subprocess, sys
try:
    from IPython.display import display, Image
except ImportError:                                  # only when run outside a notebook
    display, Image = print, (lambda f: f)
assert TRACK in ("grading", "segmentation"), "TRACK must be 'grading' or 'segmentation'"

def sh(cmd):
    """Run a shell command, stream its output (progress-bar noise removed), stop on error."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "%|" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

REPO = "/kaggle/working/DDR-DR"
if not os.path.isdir(REPO):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO}")
os.chdir(REPO)
print("working directory:", os.getcwd())

# torch/numpy/pandas/scikit-learn/matplotlib are preinstalled on Kaggle: install only what is missing.
# (Do NOT pip install -r requirements.txt here: it would replace Kaggle's torch.)
sh("pip install -q timm segmentation-models-pytorch")

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Set Accelerator to GPU in Session options (fine for QUICK_TEST, far too slow for the real run).")

PREFIX = "grading" if TRACK == "grading" else "seg"
CONFIG_GLOB = "configs/grading/g*.yaml" if TRACK == "grading" else "configs/segmentation/s*.yaml"
RUNS_DIR = "outputs/quicktest" if QUICK_TEST else "outputs/runs"

In [ ]:
# ---------------------------- PREFLIGHT: can the pretrained weights be downloaded? ----------------------------
if PRETRAINED:
    try:
        if TRACK == "grading":
            import timm; timm.create_model("efficientnet_b0", pretrained=True)
        else:
            import segmentation_models_pytorch as smp; smp.Unet("resnet34", encoder_weights="imagenet", classes=4)
        print("pretrained weights: OK")
    except Exception as e:
        raise RuntimeError("Could not download the pretrained weights. Turn Internet ON in Session options "
                           "(or set PRETRAINED = False, which gives a weaker, non-comparable-to-paper setup).") from e

In [ ]:
# ---------------------------- RESTORE from a previous version of this notebook (if attached as input) ----------------------------
def find_previous():
    for pattern in ("/kaggle/input/*/DDR-DR", "/kaggle/input/*/*/DDR-DR", "/kaggle/input/*/*/*/DDR-DR"):
        for d in glob.glob(pattern):
            if os.path.isdir(os.path.join(d, "data", "splits")) or os.path.isdir(os.path.join(d, "outputs")):
                return d
    return None

prev = find_previous()
if prev is None:
    print("No previous notebook output attached: starting fresh.")
else:
    print("Restoring from", prev)
    for sub in ("data/splits", "data/processed", "outputs/runs", "outputs/figures"):
        src, dst = os.path.join(prev, sub), os.path.join(REPO, sub)
        if os.path.isdir(src):
            os.makedirs(dst, exist_ok=True)
            sh(f"cp -rn '{src}/.' '{dst}/'")      # -n: never overwrite what is already here

In [ ]:
# ---------------------------- LOCATE THE DATASET AND INSPECT IT (Phase 1) ----------------------------
def find_data():
    want = "ddrdataset" if TRACK == "grading" else "ddr-segmentation"
    for depth in ("*", "*/*", "*/*/*"):
        for d in sorted(glob.glob(f"/kaggle/input/{depth}")):
            if os.path.isdir(d) and want in os.path.basename(d).lower():
                return d
    raise FileNotFoundError(f"No input folder containing '{want}' under /kaggle/input. Add the dataset via Add Input, "
                            "or set DATA_SRC yourself. Inputs found: " + str(os.listdir("/kaggle/input")))

SRC = DATA_SRC or find_data()
print("dataset folder:", SRC)
sh(f"python -m src.prepare inspect --src '{SRC}'")

**Check the two tables printed above** before trusting the run: grading should show about 13.7k images with grades 0-5 (and train/val/test if the dataset ships a split);
segmentation should show about 757 images with a mask count per lesion. If they are empty or clearly wrong, the folder layout differs from what `src/prepare.py` expects
(see `GUIDE.md`, Step 1) and the cells below should not be run until it is fixed.

In [ ]:
# ---------------------------- EXPLORE (Phase 1): figures for the thesis dataset chapter ----------------------------
sh(f"python -m src.explore {TRACK} --src '{SRC}'")
for f in sorted(glob.glob("outputs/figures/*.png")):
    print(f); display(Image(f))

In [ ]:
# ---------------------------- PREPROCESS + FREEZE SPLITS (Phase 2) ----------------------------
import pandas as pd
splits = [f"data/splits/{PREFIX}_{s}.csv" for s in ("train", "val", "test")]
if all(os.path.exists(p) for p in splits):
    print("Frozen splits already exist: NOT regenerating them.")
    first = pd.read_csv(splits[0])["image_path"].iloc[0]
    if not os.path.exists(first):
        print("...but the preprocessed images are missing: rebuilding the cache only (splits go to a scratch folder).")
        sh(f"python -m src.prepare {TRACK} --src '{SRC}' --splits-dir data/splits_rebuild")
        for p in splits:
            same = pd.read_csv(p).equals(pd.read_csv(p.replace("data/splits/", "data/splits_rebuild/")))
            print(("IDENTICAL " if same else "DIFFERENT (!) "), p)
else:
    sh(f"python -m src.prepare {TRACK} --src '{SRC}'")
    print("\nIMPORTANT: download data/splits/*.csv from the Output tab after saving, and commit them to GitHub. They must never be regenerated.")
for p in splits:
    print(p, len(pd.read_csv(p)), "rows")

In [ ]:
# ---------------------------- TRAIN EVERY STRATEGY (Phases 3-4) ----------------------------
# Only the imbalance setting differs between runs; everything else comes from configs/<track>/base.yaml.
overrides = [f"train.out_dir={RUNS_DIR}", f"train.num_workers={NUM_WORKERS}", f"model.pretrained={'true' if PRETRAINED else 'false'}"]
if QUICK_TEST:
    overrides.append("train.epochs=1")
configs = sorted(glob.glob(CONFIG_GLOB))
print(f"{len(configs)} runs:", [os.path.basename(c) for c in configs])
for cfg in configs:
    sh(f"python -m src.train --config {cfg} --set {' '.join(overrides)}")     # finished runs are skipped automatically

In [ ]:
# ---------------------------- COMPARE ON VALIDATION (Phase 5) ----------------------------
sh(f"python -m src.compare --track {TRACK} --split val --runs {RUNS_DIR} --out {RUNS_DIR}/..")
try:
    display(pd.read_csv(f"outputs/comparison_{TRACK}_val.csv", index_col=0))
    display(Image(f"outputs/comparison_{TRACK}_val.png"))
except Exception as e:
    print("(could not display the table/chart:", e, ")")

## What to do next (not run automatically)

1. **Save the results.** *Save Version -> Save & Run All (Commit)* (or Quick Save). Download `data/splits/*.csv` and the `config.yaml / history.csv / val_metrics.json / train.log`
   of each run from the Output tab and commit them to GitHub.
2. **Check noise.** Repeat the top 2-3 strategies *and the baseline* with seeds 43 and 44 (e.g. `python -m src.train --config configs/grading/g2_oversample.yaml --seed 43 --set train.num_workers=2`),
   then re-run the compare cell: it reports mean +/- std.
3. **Optional combined run:** edit `configs/<track>/combined.yaml` to the best sampler/loss and train it.
4. **Pick one winner per track and write down why in the README before using the test split.**
5. **Test once** (a new cell, only now): `!python -m src.evaluate --run outputs/runs/<winner> --split test` and the same for the baseline `g0_baseline` / `s0_baseline`.
6. **External sets:** see `KAGGLE_GUIDE.md`, Step 7.